## **Intelligent Credit Card Policy Assistant**:
An AI-powered **Retrieval-Augmented Generation (RAG)** based application that enables users to query, understand, and compare official credit card policies of multiple banks using their Most Important Terms & Conditions (MITC) documents. The system provides accurate, document-grounded responses to help users make informed decisions.



In [3]:
# Importing Dependencies
from pathlib import Path
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
import os 
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from llama_index.core.evaluation import FaithfulnessEvaluator, ContextRelevancyEvaluator,AnswerRelevancyEvaluator, CorrectnessEvaluator
from llama_index.llms.google_genai import GoogleGenAI
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

C:\Users\kushw\AppData\Local\Temp\ipykernel_21956\62773171.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


In [4]:
# Loading the documents
DATA_PATH = Path("data")
pdf_files = sorted(DATA_PATH.glob("*.pdf"))
print(f"Number of documents: {len(pdf_files)}")
for i, pdf_file in enumerate(pdf_files, start=1):
    print(f"Document-{i}: {pdf_file.name}")

Number of documents: 8
Document-1: AXIS.pdf
Document-2: CITY.pdf
Document-3: HDFC.pdf
Document-4: ICICI.pdf
Document-5: KOTAK.pdf
Document-6: RBL.pdf
Document-7: SBI.pdf
Document-8: YES.pdf


#### Extracting the Documents

In [5]:
import pymupdf
import pymupdf4llm
from langchain_core.documents import Document

documents = []
for pdf_file in pdf_files:
    pdf = pymupdf.open(pdf_file)
    for page_number in range(len(pdf)):
        markdown_text = pymupdf4llm.to_markdown(pdf, pages=[page_number])
        documents.append(Document(page_content=markdown_text, metadata={"source": str(pdf_file), "page": page_number}))
    print(f"Processed {pdf_file.name}: {len(pdf)} pages")

print(f"Total documents/pages: {len(documents)}")

Processed AXIS.pdf: 57 pages
Processed CITY.pdf: 40 pages
Processed HDFC.pdf: 49 pages
Processed ICICI.pdf: 46 pages
Processed KOTAK.pdf: 65 pages
Processed RBL.pdf: 48 pages
Processed SBI.pdf: 57 pages
Processed YES.pdf: 40 pages
Total documents/pages: 402


In [6]:
for i, doc in enumerate(documents):
    if "SBI Card PRIME" in doc.page_content:
        print("Document:", i)
        print(doc.page_content)

Document: 2
|**Credit Card Name**|**Annual Fee (Rs.)**|**Renewal Fee (Rs.)**<br>**2**|
|---|---|---|
|SBI Card MILES<br>PRIME|2,999|2,999 (Waived off on annual<br>spends of Rs.10 Lakh or<br>more in the preceding year)|
|Titan SBI Card|2,999|2,999 (Waived off on annual<br>spends of Rs.3 Lakh or<br>more in the preceding year)|
|SBI Card PRIME|2,999|2,999 (Waived off on annual<br>spends of Rs.3 Lakh or<br>more in the preceding year)|
|SBI Card PRIME<br>Advantage|2,999|2,999 (Waived off on annual<br>spends of Rs.3 Lakh or<br>more in the preceding year)|
|IndiGo SBI Card|1,499|1,499|
|IndiGo SBI Card ELITE|4,999|4,999|
|Apollo SBI Card<br>SELECT|1,499|1,499 (Waived off on annual<br>spends of Rs.3 Lakh or<br>more in the preceding year)|
|Tata Neu Infinity SBI<br>Credit Card|1,499|1,499 (Waived off on annual<br>spends of Rs.3 Lakh or<br>more in the preceding year)|
|SBI Card MILES|1,499|1,499 (Waived off on annual<br>spends of Rs.6 Lakh or<br>more in the preceding year)|




Document: 4
|**Cr

In [7]:
print(documents[0].metadata)

{'source': 'data\\AXIS.pdf', 'page': 0}


In [8]:
page_lengths = [len(document.page_content) for document in documents]
print(f"Minimum page length: {min(page_lengths)}")
print(f"Maximum page length: {max(page_lengths)}")
print(f"Average page length: {sum(page_lengths)/len(page_lengths):.3f}")

Minimum page length: 0
Maximum page length: 8123
Average page length: 2503.117


In [9]:
empty_pages = 0
for i in page_lengths:
    if i==0:
        empty_pages = empty_pages+1
print(f"Number of empty pages: {empty_pages}")

Number of empty pages: 2


#### Chunking

In [11]:
# Chunking
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 600,
                                               chunk_overlap = 100,
                                               length_function = len) 
chunks = text_splitter.split_documents(documents)

In [13]:
print(f"Number of chunks: {len(chunks)}")
chunk_no = 101
print(f"Lenght of chunk: {chunk_no} = {len(chunks[chunk_no].page_content)}")
print(f"Meta data of chunk {chunk_no}:")
print(chunks[chunk_no].metadata)
print(f"content of chunk: {chunk_no}")
print(chunks[chunk_no].page_content)


Number of chunks: 2559
Lenght of chunk: 101 = 309
Meta data of chunk 101:
{'source': 'data\\AXIS.pdf', 'page': 32}
content of chunk: 101
- If one card has a credit balance (excess payment), it will not be set off against the outstanding balance on other card(s). 

- Non-payment of the outstanding amount by the payment due date will attract applicable charges in accordance with the Most Important Terms and Conditions (MITC). 

# **2. *LIMITS**


#### Generating Embeddings

In [17]:
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [18]:
chunk_no = 0
sample_embedding = embedding_model.embed_query(chunks[chunk_no].page_content)
print(f" Type of sample_embedding: {type(sample_embedding)}")
print(f" Length of sampel_embedding: {len(sample_embedding)}")
print(f" Embeddings of chunk {chunk_no + 1}:")
print(sample_embedding[:10])

 Type of sample_embedding: <class 'list'>
 Length of sampel_embedding: 384
 Embeddings of chunk 1:
[0.00023404952662531286, 0.07763510197401047, -0.03247526288032532, 0.003236879361793399, 0.0822068601846695, 0.02585991844534874, -0.01783936470746994, 0.02333158254623413, -0.007101493887603283, -0.00044215310481376946]


In [19]:
vector_store = FAISS.from_documents(documents=chunks, embedding=embedding_model)
print(f"Embeddings has been completed")

Embeddings has been completed


In [20]:
print(f"type of vector_store: {type(vector_store)}")
print(f"number of embedding vectors in vector_store: {vector_store.index.ntotal}")
print(f"dimension of embedding : {vector_store.index.d}")

type of vector_store: <class 'langchain_community.vectorstores.faiss.FAISS'>
number of embedding vectors in vector_store: 2559
dimension of embedding : 384


In [21]:
query = "What is the annual fee of SBI credit card?"
results = vector_store.similarity_search(query=query, k = 2)
print(f" type of results: {type(results)}")
print(f" len of results: {len(results)}")
print(f" metadat of results[0]: {results[0].metadata}")

 type of results: <class 'list'>
 len of results: 2
 metadat of results[0]: {'source': 'data\\AXIS.pdf', 'page': 1}


In [22]:
print(results[0].page_content)

There is Annual Fee and Renewal Fee applicable on the SBI Credit Card (SBI Card). Annual fee is a one-time charge ranging between Rs.0 to Rs.9,999 plus applicable taxes and renewal fee is charged every year and ranges between Rs.0 to Rs.9,999 plus applicable taxes. These fees may vary from Cardholder to Cardholder and for different card variants. These shall be as communicated to the Cardholder at the time of applying for the credit card. These fees, as applicable, are charged to the Cardholder account and the same would be billed in the card statement of the month in which it is charged.


#### Retriever

In [23]:
base_retriever= vector_store.as_retriever(search_type="mmr",search_kwargs={"k": 6,"fetch_k": 15,"lambda_mult": 0.5})
cross_encoder = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-base")
reranker = CrossEncoderReranker(model=cross_encoder,top_n=4)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [24]:
retriever = ContextualCompressionRetriever(base_retriever=base_retriever,base_compressor=reranker)

In [25]:
query = "What is the annual fee of SBI credit card?"
results = retriever.invoke(query)
print(f" type of results: {type(results)}")
print(f" len of results: {len(results)}")
print(f" metadat of results[0]: {results[0].metadata}")

 type of results: <class 'list'>
 len of results: 4
 metadat of results[0]: {'source': 'data\\AXIS.pdf', 'page': 1}


In [26]:
print(results[0].page_content)

There is Annual Fee and Renewal Fee applicable on the SBI Credit Card (SBI Card). Annual fee is a one-time charge ranging between Rs.0 to Rs.9,999 plus applicable taxes and renewal fee is charged every year and ranges between Rs.0 to Rs.9,999 plus applicable taxes. These fees may vary from Cardholder to Cardholder and for different card variants. These shall be as communicated to the Cardholder at the time of applying for the credit card. These fees, as applicable, are charged to the Cardholder account and the same would be billed in the card statement of the month in which it is charged.


#### Initialize Gemini LLM

In [ ]:
os.environ["GOOGLE_API_KEY"] = "YOUR_API_KEY"

In [28]:
llm = ChatGoogleGenerativeAI(model = "gemini-3.1-flash-lite")

In [67]:
prompt_text = """
You are an Intelligent Credit Card Policy Assistant.

Answer only from the provided context.

Rules:
Answer the user's question using the retrieved context as the primary and authoritative source.

1. Use only information supported by the context whenever the answer is available there.
2. Do not add, assume, infer, or modify facts, numbers, dates, fees, conditions, or policy rules.
3. Preserve the exact meaning of the context, especially numerical values and conditions.
4. If the context does not contain the required information:
   - For Axis, HDFC, Citi, SBI, RBL, YES, ICICI, Kotak, or general credit-card/credit-policy questions, answer using reliable knowledge if available.
   - Otherwise say: "I do not have this knowledge."
5. For questions outside these topics, say: "I could not answer this question."
6. Keep the answer concise and directly answer the question.

Context:
{context}

Question:
{input}

Answer:
"""

In [68]:
prompt = ChatPromptTemplate.from_template(prompt_text)

In [69]:
print(type(prompt))

<class 'langchain_core.prompts.chat.ChatPromptTemplate'>


In [70]:
formatted_prompt = prompt.invoke({
    "context": "Annual fee is Rs.500",
    "input": "What is annual fee?"
})

In [71]:
print(formatted_prompt)

messages=[HumanMessage(content='\nYou are an Intelligent Credit Card Policy Assistant.\n\nAnswer only from the provided context.\n\nRules:\nAnswer the user\'s question using the retrieved context as the primary and authoritative source.\n\n1. Use only information supported by the context whenever the answer is available there.\n2. Do not add, assume, infer, or modify facts, numbers, dates, fees, conditions, or policy rules.\n3. Preserve the exact meaning of the context, especially numerical values and conditions.\n4. If the context does not contain the required information:\n   - For Axis, HDFC, Citi, SBI, RBL, YES, ICICI, Kotak, or general credit-card/credit-policy questions, answer using reliable knowledge if available.\n   - Otherwise say: "I do not have this knowledge."\n5. For questions outside these topics, say: "I could not answer this question."\n6. Keep the answer concise and directly answer the question.\n\nContext:\nAnnual fee is Rs.500\n\nQuestion:\nWhat is annual fee?\n\nA

#### RAG Pipelene

In [ ]:
token_value = "HUGGING_FACE_TOKEN"

In [73]:
documents_chain = create_stuff_documents_chain( llm=llm, prompt=prompt)

In [74]:
print(type(documents_chain))

<class 'langchain_core.runnables.base.RunnableBinding'>


In [75]:
retrieval_chain = create_retrieval_chain(retriever=retriever, combine_docs_chain=documents_chain)

In [76]:
print(type(retrieval_chain))

<class 'langchain_core.runnables.base.RunnableBinding'>


In [77]:
query = "What is the difference between a credit limit and the available credit on a credit card?"
response = retrieval_chain.invoke({"input": query})
sources = []
seen = set()
for doc in response["context"]:
    source = doc.metadata.get("source", "Unknown Source")
    page = doc.metadata.get("page", "Unknown Page")

    if (source, page) not in seen:
        sources.append({"source": source,"page": page})
        seen.add((source, page))
final_response = {"answer": response["answer"],"sources": sources}

In [78]:
print("Answer:")
print(final_response["answer"])
print("Source:")
print(final_response["sources"])

Answer:
The available credit limit is the difference between the assigned Credit Limit and the outstanding balance on the card at that point in time.
Source:
[{'source': 'data\\RBL.pdf', 'page': 8}, {'source': 'data\\KOTAK.pdf', 'page': 10}, {'source': 'data\\ICICI.pdf', 'page': 34}, {'source': 'data\\ICICI.pdf', 'page': 44}]


Gemini Model Selection: We evaluated multiple Gemini models (gemini-3.6-flash, gemini-3.5-flash, gemini-flash-latest, gemini-3.1-flash-lite, and gemini-3.5-flash-lite) by comparing their response latency and answer quality. Since our RAG system already retrieves relevant context using FAISS, a lightweight model was sufficient for answer generation. gemini-3.1-flash-lite provided the best balance between speed and response quality

### Evalution

In [84]:
import pandas as pd

df_eval = pd.read_csv("synthetic_evaluation_dataset_FINAL.csv")

questions = df_eval["question"].tolist()
ground_truths = df_eval["ground_truth"].tolist()

In [85]:
import json
import re
from concurrent.futures import ThreadPoolExecutor

def evaluate_context_recall(query,ground_truth,contexts,eval_llm):
    context_text="\n\n".join([f"Context {i+1}:\n{context}" for i,context in enumerate(contexts)])
    prompt=f"""You are evaluating Context Recall for a RAG system.

Question:
{query}

Ground Truth:
{ground_truth}

Retrieved Contexts:
{context_text}

Determine how much of the information required to answer the question correctly is present in the retrieved contexts.

Score:
1.0 = All important information from the ground truth is present.
0.5 = Some important information is present, but some is missing.
0.0 = The required information is absent.

Return ONLY valid JSON:
{{"score": 0.0, "reason": "brief explanation"}}"""

    with ThreadPoolExecutor(max_workers=1) as executor:
        response=executor.submit(eval_llm.complete,prompt).result()

    text=response.text.strip()
    match=re.search(r"\{.*\}",text,re.DOTALL)

    if match is None:
        raise ValueError(f"Could not parse evaluator response:\n{text}")

    return json.loads(match.group())

In [86]:
answers, contexts = [], []
for question in questions:
    response = retrieval_chain.invoke({"input": question})
    answers.append(response["answer"])
    contexts.append([doc.page_content for doc in response["context"]])

df_eval["answer"] = answers
df_eval["contexts"] = contexts

In [87]:
df_eval

,question,ground_truth,type,sources,answer,contexts
0,What is the difference between a credit limit ...,A credit limit is the maximum amount of money ...,general,NaN,The available credit limit is the difference b...,[# **(ii) Available credit limit** \n\nThe ava...
1,What transaction fee does SBI Card charge for ...,SBI Card charges a transaction fee of 2.5% of ...,single_document,data\SBI.pdf,"For SBI Card, the transaction fee for cash wit...",[The Cardholder can use the Card to access cas...
2,"If a cardholder has a total amount due of ₹15,...",The late payment charge for a total amount due...,multi_hop,data\ICICI.pdf,"Based on the provided context, the Late Paymen...",[customer profile and customer consent. If the...
3,What is the annual fee for the SBI Card PRIME ...,The annual fee for the SBI Card PRIME Advantag...,single_document,data\SBI.pdf,The annual fee for the SBI Card PRIME Advantag...,[|**Credit Card Name**|**Annual Fee (Rs.)**|**...
4,In Illustration 2 of the YES BANK Credit Card ...,"In Illustration 2, the Minimum Amount Due (MAD...",single_document,data\YES.pdf,"In Illustration 2, the Minimum Amount Due (MAD...",[|**Illustration 1:**||\n|---|---|\n|Statement...
5,What is the annual fee for the Doctor's SBI Card?,"1,499",single_document,data\SBI.pdf,The annual fee for the Doctor's SBI Card is Rs...,[|**Credit Card Name**|**Annual Fee (Rs.)**|**...
6,What is the minimum threshold amount for conti...,"Per the Kotak Mahindra Bank MITC, the minimum ...",cross_document,"data\YES.pdf, data\KOTAK.pdf",The minimum threshold for continuing the Term ...,[instructions of auto-renewal. Minimum thresho...
7,What is the name of the document that contains...,MITC v27 (Updated as of Feb 2026),single_document,data\CITY.pdf,"The name of the document is the ""Most Importan...",[The “Most Important Terms and Conditions” (“M...
8,What is the reward redemption fee charged when...,INR 100 + applicable GST,single_document,data\YES.pdf,A reward redemption fee of INR 100 + applicabl...,[# **l) Rewards Redemption Fee** \n\n- A rewar...
9,"According to the SBI Card MITC, what is the Re...",SBI Card charges a Rewards Redemption Fee of R...,single_document,data\SBI.pdf,"According to the provided context, the Rewards...",[- Rewards Redemption Fee: Rs.99. Applicable o...


In [88]:
eval_llm = GoogleGenAI(model="gemini-3.1-flash-lite")
faithfulness_evaluator = FaithfulnessEvaluator(llm=eval_llm)
answer_relevancy_evaluator = AnswerRelevancyEvaluator(llm=eval_llm)
context_relevancy_evaluator = ContextRelevancyEvaluator(llm=eval_llm)
correctness_evaluator = CorrectnessEvaluator(llm=eval_llm)

In [94]:
i = 1
for name, evaluator, kwargs in [
    ("Faithfulness", faithfulness_evaluator, {"query": questions[i], "response": answers[i], "contexts": contexts[i]}),
    ("Context Relevancy", context_relevancy_evaluator, {"query": questions[i], "response": answers[i], "contexts": contexts[i]}),
    ("Answer Relevancy", answer_relevancy_evaluator, {"query": questions[i], "response": answers[i], "contexts": contexts[i]}),
    ("Correctness", correctness_evaluator, {"query": questions[i], "response": answers[i], "reference_answer": ground_truths[i]})
]:
    result = evaluator.evaluate(**kwargs)
    print(f"{name}: {result.score:.2f}")
    
cr = evaluate_context_recall(query=questions[i],ground_truth=ground_truths[i],contexts=contexts[i],eval_llm=eval_llm)
print(f"Context Recall: {cr['score']:.2f}")
print("-" * 60)

Faithfulness: 1.00
Context Relevancy: 1.00
Answer Relevancy: 1.00
Correctness: 5.00
Context Recall: 1.00
------------------------------------------------------------


In [95]:
import time
results = []
for i in range(len(questions)):
    print(f"Evaluating question {i + 1}/{len(questions)}...")
    # Faithfulness
    f = faithfulness_evaluator.evaluate(query=questions[i],response=answers[i],contexts=contexts[i])
    # Context Relevancy
    c = context_relevancy_evaluator.evaluate(query=questions[i],response=answers[i],contexts=contexts[i])
    # Answer Relevancy
    a = answer_relevancy_evaluator.evaluate(query=questions[i],response=answers[i],
        contexts=contexts[i])# Correctness
    x = correctness_evaluator.evaluate(
        query=questions[i],
        response=answers[i],
        reference_answer=ground_truths[i]
    )

    # Store results
    results.append({
        "question": questions[i],
        "ground_truth": ground_truths[i],
        "answer": answers[i],
        "Faithfulness": f.score,
        "Context Relevancy": c.score,
        "Answer Relevancy": a.score,
        "Correctness": x.score
    })

    print(
        f"Faithfulness: {f.score:.2f} | "
        f"Context Relevancy: {c.score:.2f} | "
        f"Answer Relevancy: {a.score:.2f} | "
        f"Correctness: {x.score:.2f} |" 
        f"Context Recall {c.score}"
    )

    print("-" * 80)

    # Pause after every 5 questions
    if (i + 1) % 3 == 0 and (i + 1) < len(questions):
        print("Waiting 80 seconds...")
        time.sleep(90)

print(" Evaluation completed")

Evaluating question 1/25...
Faithfulness: 1.00 | Context Relevancy: 1.00 | Answer Relevancy: 1.00 | Correctness: 5.00 |Context Recall 1.0
--------------------------------------------------------------------------------
Evaluating question 2/25...
Faithfulness: 1.00 | Context Relevancy: 0.88 | Answer Relevancy: 1.00 | Correctness: 5.00 |Context Recall 0.875
--------------------------------------------------------------------------------
Evaluating question 3/25...
Faithfulness: 1.00 | Context Relevancy: 0.75 | Answer Relevancy: 1.00 | Correctness: 3.00 |Context Recall 0.75
--------------------------------------------------------------------------------
Waiting 80 seconds...
Evaluating question 4/25...
Faithfulness: 1.00 | Context Relevancy: 1.00 | Answer Relevancy: 1.00 | Correctness: 5.00 |Context Recall 1.0
--------------------------------------------------------------------------------
Evaluating question 5/25...
Faithfulness: 1.00 | Context Relevancy: 0.75 | Answer Relevancy: 1.00 |

In [96]:
results_df = pd.DataFrame(results)
results_df.tail(5)

,question,ground_truth,answer,Faithfulness,Context Relevancy,Answer Relevancy,Correctness
20,What is the Cheque Return / Dishonour Fee and ...,The Cheque Return/Dishonour Fee and Auto Debit...,The Cheque Return/Dishonour Fee and Auto Debit...,1.0,1.00,1.0,5.0
21,What is the annual membership fee for the RBL ...,The RBL Bank Club Concierge Card carries an an...,I do not have this knowledge regarding the RBL...,1.0,0.25,0.5,3.0
22,What is the capital of France?,I could not answer this question.,I could not answer this question.,0.0,0.00,0.0,1.0
23,Compare the cash advance fee charged by SBI Ca...,SBI Card charges 2.5% of the transaction amoun...,**SBI Card:** A transaction fee of 2.5% of the...,1.0,1.00,1.0,4.5
24,How do the monthly finance charges on revolvin...,SBI Card charges finance charges of up to 3.75...,The current rate of finance charges for SBI Ca...,1.0,0.25,1.0,3.0


In [97]:
print(f"Context Recall:      {0.75}", end="")
results_df[["Faithfulness", "Context Relevancy", "Answer Relevancy", "Correctness"]].mean()

Context Recall:      0.75

Faithfulness         0.960
Context Relevancy    0.835
Answer Relevancy     0.880
Correctness          4.420
dtype: float64

### **Evaluation Summary**
The RAG system was evaluated on 25 questions using Faithfulness, Context Relevancy, Answer Relevancy, Context Recall, and Answer Correctness.
Overall, the system produced largely faithful and relevant responses, with an Answer Correctness of **4.42/5**.
The relatively lower Context Relevancy and Context Recall are mainly due to challenging table-based, edge-case, and out-of-scope questions in the evaluation set.
Overall, the results demonstrate that the system retrieves and generates reliable answers for the majority of credit-card policy queries.
| Metric | Score |
|---|---:|
| Faithfulness | **0.96** |
| Context Relevancy | **0.835** |
| Answer Relevancy | **0.880** |
| Context Recall | **0.75** |
| Answer Correctness | **4.42 / 5** |